# Business Location CSV 생성

원본 `yelp_academic_dataset_business.json`에서 피처 엔지니어링에 필요한 위치 정보만 추출합니다.

생성 파일:

```text
business_location.csv
```

포함 컬럼:

```text
business_id, city, state, latitude, longitude
```

## 1. 경로 설정

로컬에서는 기본적으로 `team_project/archive/yelp_academic_dataset_business.json`을 찾습니다. Colab에서 실행하려면 Google Drive에 JSON을 올린 뒤 `BUSINESS_JSON_PATH`만 해당 경로로 바꾸면 됩니다.

In [ ]:
from pathlib import Path
import pandas as pd


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ModuleNotFoundError:
        return False


IN_COLAB = running_in_colab()

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project/Team-6')
    BUSINESS_JSON_PATH = PROJECT_ROOT / 'data' / 'yelp_academic_dataset_business.json'
else:
    PROJECT_ROOT = Path.cwd().resolve()
    for path in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if path.name == 'Team-6':
            PROJECT_ROOT = path
            break

    BUSINESS_JSON_PATH = PROJECT_ROOT.parent / 'archive' / 'yelp_academic_dataset_business.json'

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'interim'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'business_location.csv'

print(f'IN_COLAB: {IN_COLAB}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'BUSINESS_JSON_PATH: {BUSINESS_JSON_PATH}')
print(f'OUTPUT_PATH: {OUTPUT_PATH}')

## 2. 원본 JSON 확인

In [ ]:
if not BUSINESS_JSON_PATH.exists():
    raise FileNotFoundError(
        'business JSON file was not found. '\
        'Check BUSINESS_JSON_PATH and update it to the actual file path.\n'
        f'BUSINESS_JSON_PATH: {BUSINESS_JSON_PATH}'
    )

print('business JSON file exists.')
print(f'file size MB: {BUSINESS_JSON_PATH.stat().st_size / 1024 / 1024:.2f}')

## 3. 위치 정보 추출

In [ ]:
USE_COLUMNS = ['business_id', 'city', 'state', 'latitude', 'longitude']

business_df = pd.read_json(BUSINESS_JSON_PATH, lines=True)
business_location = business_df[USE_COLUMNS].copy()

business_location = business_location.dropna(subset=['business_id'])
business_location = business_location.drop_duplicates('business_id')

print('business_location shape:', business_location.shape)
display(business_location.head())

## 4. 저장 및 검증

In [ ]:
business_location.to_csv(OUTPUT_PATH, index=False)

saved_df = pd.read_csv(OUTPUT_PATH)
print(f'Saved: {OUTPUT_PATH}')
print('saved shape:', saved_df.shape)
print('missing values:')
print(saved_df.isnull().sum())
display(saved_df.head())